In [2]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
stocks_name = ['BTC-USD', 'ETH-USD', 'SPY', 'GLD', '^VIX']

In [4]:
stocks_data = yf.download(stocks_name, start='2018-01-01', end=pd.Timestamp.now().date(), interval='1mo')

[*********************100%***********************]  5 of 5 completed


In [5]:
df_stocks = pd.DataFrame(stocks_data).dropna()

In [6]:
df_stocks_close = df_stocks['Close'].dropna().drop(columns=['^VIX']).copy()

Copy the DataFrame to a new variable

In [7]:
df_stocks_close_criptos = df_stocks['Close'].copy()

Thechnical Indicators

In [8]:
###
# Calculate SMAs for BTC-USD
df_stocks_close_criptos['BTC-USD_EMA-20'] = df_stocks_close_criptos['BTC-USD'].ewm(span=20).mean()
df_stocks_close_criptos['BTC-USD_EMA-50'] = df_stocks_close_criptos['BTC-USD'].ewm(span=50).mean()
df_stocks_close_criptos['BTC-USD_EMA-200'] = df_stocks_close_criptos['BTC-USD'].ewm(span=200).mean()
###
# Calculate SMAs for ETH-USD
df_stocks_close_criptos['ETH-USD_EMA-20'] = df_stocks_close_criptos['ETH-USD'].ewm(span=20).mean()
df_stocks_close_criptos['ETH-USD_EMA-50'] = df_stocks_close_criptos['ETH-USD'].ewm(span=50).mean()
df_stocks_close_criptos['ETH-USD_EMA-200'] = df_stocks_close_criptos['ETH-USD'].ewm(span=200).mean()
###

MACD

In [9]:
###
#MACD BTC-USD
def calculate_macd(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['BTC-USD'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['BTC-USD'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd, signal, histogram = calculate_macd(df_stocks)

In [10]:
###
#MACD SPY
def calculate_macd_spy(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['SPY'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['SPY'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd_spy, signal_spy, histogram_spy = calculate_macd_spy(df_stocks)

Bollinger Bands

In [11]:
window = 20
std_dev = 2

price = df_stocks["Close"]["BTC-USD"]
middle = price.rolling(window).mean()
std = price.rolling(window).std()
upper = middle + std_dev * std
lower = middle - std_dev * std

bb_percent_b = (price - lower) / (upper - lower)

In [12]:
window_spy = 20
std_dev_spy = 2

price_spy = df_stocks["Close"]["SPY"]
middle_spy = price_spy.rolling(window_spy).mean()
std_spy = price_spy.rolling(window_spy).std()
upper_spy = middle_spy + std_dev_spy * std_spy
lower_spy = middle_spy - std_dev_spy * std_spy

bb_percent_spy = (price_spy - lower_spy) / (upper_spy - lower_spy)

Stochastic + RSI

In [13]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=2, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para BTC-USD
rsi = calculate_rsi(df_stocks['Close']['BTC-USD'])
stoch_k, stoch_d = calculate_stochastic(
    df_stocks['High']['BTC-USD'],
    df_stocks['Low']['BTC-USD'],
    df_stocks['Close']['BTC-USD']
)

Returns of the investments in multiplication

In [14]:
stocks_returns_month_df = (df_stocks_close.pct_change().dropna())#Day variation
acumulated_returns = (1 + stocks_returns_month_df).cumprod()

Log Return

In [15]:
stocks_returns_log_df = np.log(df_stocks_close/ df_stocks_close.shift(1)).dropna()

Volatility month Sharpe & Sortino

In [16]:
# Volatilidade / janelas (dados mensais)
# Observação: a anualização para mensal deve usar 12 (meses por ano),
# independente do tamanho da janela usada no rolling.
trading_month_sharpe = 12 #6
trading_month_sortino = 18 #14

periods_per_year = 12

# Volatilidade anualizada (usada no Sharpe)
volatility_month_sharpe = (
    stocks_returns_log_df.rolling(window=trading_month_sharpe).std() * np.sqrt(periods_per_year)
)

# (opcional) Mantém a variável para consistência com o resto do notebook
volatility_month_sortino = (
    stocks_returns_log_df.rolling(window=trading_month_sortino).std() * np.sqrt(periods_per_year)
)

Sharpe Ratio Month(12)

In [17]:
# Sharpe Ratio (dados mensais)
# rf deve ser taxa livre de risco ANUAL (ex: 0.01 = 1% a.a.)
rf = 0.01

periods_per_year = 12

rolling_mean_annual = stocks_returns_log_df.rolling(window=trading_month_sharpe).mean() * periods_per_year
rolling_std_annual = stocks_returns_log_df.rolling(window=trading_month_sharpe).std() * np.sqrt(periods_per_year)

sharpe_ratio = (rolling_mean_annual - rf) / rolling_std_annual

Sortino Ratio

In [18]:
# Sortino Ratio (dados mensais)
# Downside deviation: desvio padrão apenas do retorno abaixo do MAR (aqui, MAR = rf/12)
rf = 0.01
periods_per_year = 12
mar = rf / periods_per_year

# Retornos abaixo do MAR (mantém apenas a parte negativa em relação ao MAR)
downside = (stocks_returns_log_df - mar).clip(upper=0)

downside_deviation_annual = downside.rolling(window=trading_month_sortino).std() * np.sqrt(periods_per_year)
rolling_mean_annual = stocks_returns_log_df.rolling(window=trading_month_sortino).mean() * periods_per_year

sortino_ratio = (rolling_mean_annual - rf) / downside_deviation_annual

Data Conditioning of BTC and SPY

In [19]:
# %% [markdown]
# ## Data Conditioning BTC

nv_shp_btc_top = 1.75
nv_srt_btc_top = 4.5
nv_shp_btc_bottom = -1.5
nv_srt_btc_bottom = -1.2
mask_btc_sortino_top = sortino_ratio['BTC-USD'] >= 4.5
mask_btc_sharpe_top = sharpe_ratio['BTC-USD'] > nv_shp_btc_top
mask_btc_sharpe_sortino_top = ((sortino_ratio['BTC-USD'] >= nv_srt_btc_top) & mask_btc_sharpe_top)
mask_btc_sortino_reversion_down = (sharpe_ratio['BTC-USD'] < 1.65) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top) | ((sharpe_ratio['BTC-USD'] > nv_shp_btc_top) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top))
mask_btc_sharpe_bottom = sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom
mask_btc_sortino_bottom = (sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom
mask_btc_sharpe_sortino_bottom = ((sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom)
mask_btc_sortino_reversion_up_down = (sharpe_ratio['BTC-USD'] > 0) & (sortino_ratio['BTC-USD'] <= -1.5) | (sharpe_ratio['BTC-USD'] < 0) & (sortino_ratio['BTC-USD'] > 1.9)

# %% [markdown]
# ## Data Conditioning SPY
# %%

nv_shp_spy_top = 2.2
nv_srt_spy_top = 3.5
nv_shp_spy_bottom = 0
nv_srt_spy_bottom = 0
mask_spy_sortino_top = sortino_ratio['SPY'] >= nv_srt_spy_top
mask_spy_sharpe_top = sharpe_ratio['SPY'] > nv_shp_spy_top
mask_spy_sharpe_sortino_top = ((sortino_ratio['SPY'] >= nv_srt_spy_top) & mask_spy_sharpe_top)
mask_spy_sortino_reversion_down = (sharpe_ratio['SPY'] < nv_shp_spy_bottom) & (sortino_ratio['SPY'] >= 3.0)
mask_spy_sharpe_bottom = sharpe_ratio['SPY'] < -0.6
mask_spy_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom)
mask_spy_sharpe_sortino_bottom = ((sortino_ratio['SPY'] < 0) & mask_spy_sharpe_bottom)
mask_spy_sortino_reversion_up = (sharpe_ratio['SPY'] > 1.0) & (sortino_ratio['SPY'] <= 0)


In [20]:
def plot_top_now():

    if (sharpe_ratio['BTC-USD'].iloc[-1] > 1.75) & (sortino_ratio['BTC-USD'].iloc[-1] > 4.5) & (bb_percent_b.iloc[-1] > 1) & (rsi.iloc[-1] < 80):
        print('BTC-USD atende aos critérios')
    else:
        print('BTC-USD não atende aos critérios')

def plot_bottom_now():

    if (sharpe_ratio['BTC-USD'].iloc[-1] < -1.5) & (sortino_ratio['BTC-USD'].iloc[-1] < -1.2) & (bb_percent_b.iloc[-1] < 0.1) & (rsi.iloc[-1] < 20):
        print('BTC-USD atende aos critérios')
    else:
        print('BTC-USD não atende aos critérios')

def plot_top():

    points_top = []
    for date in range(len(df_stocks['Close'])-1):

        if (sharpe_ratio['BTC-USD'].iloc[date] > 1.7) & (sortino_ratio['BTC-USD'].iloc[date] > 4.0) & (bb_percent_b.iloc[date] > 1) & (rsi.iloc[date] > 80):
            points_top.append(date+1)
            print('BTC-USD atende aos critérios')
    return points_top

def plot_bottom():

    points_bottom = []
    for date in range(len(df_stocks['Close'])-1):
        if (sharpe_ratio['BTC-USD'].iloc[date] < -0.7) & (sortino_ratio['BTC-USD'].iloc[date] < -0.5) & (bb_percent_b.iloc[date] < 0.3) & (rsi.iloc[date] < 26):
            points_bottom.append(date-1)
            print('BTC-USD atende aos critérios')
    return points_bottom

top_points = plot_top()
bottom_points = plot_bottom()
print(top_points)
print(bottom_points)

BTC-USD atende aos critérios
BTC-USD atende aos critérios
BTC-USD atende aos critérios
BTC-USD atende aos critérios
BTC-USD atende aos critérios
BTC-USD atende aos critérios
[38, 39, 73, 74]
[58, 59]


Graphic

In [22]:
# %% [markdown]
# ## Data Visualization
# %%

fig = make_subplots(
    rows=9,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.01,
    row_heights=[2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
)

# %% [markdown]
# ## Data Plotting Prices
# %%
fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['BTC-USD'],
    high=df_stocks['High']['BTC-USD'],
    low=df_stocks['Low']['BTC-USD'],
    close=df_stocks['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks.index[top_points],
    y=df_stocks['Close']['BTC-USD'].iloc[top_points],
    mode='markers+text',
    marker=dict(size=10,
                color='lightcoral',
                symbol='diamond'),
    name=f'Top Points ({len(top_points)} points)'),
    row=1, col=1
)

fig.add_trace(go.Scatter(
    x=df_stocks.index[bottom_points],
    y=df_stocks['Close']['BTC-USD'].iloc[bottom_points],
    mode='markers+text',
    marker=dict(size=10,
                color='lightgreen',
                symbol='diamond'),
    name=f'Bottom Points ({len(bottom_points)} points)'),
    row=1, col=1
)

###
# Plot BTC SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-20'],
    name='EMA-20',
    line=dict(color='lightgreen')
), row=1, col=1)


fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-50'],
    name='EMA-50',
    line=dict(color='lightblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-200'],
    name='EMA-200',
    line=dict(color='white')
), row=1, col=1)

# %% [markdown]
# ## Data Plotting MACD BTC
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd,
    name='MACD',
    marker_color='lightblue',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal,
    name='Signal',
    marker_color='lightcoral',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram,
), row=2, col=1)

# %% [markdown]
# ## Data Plotting MACD SPY
# %%

fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd_spy,
    name='MACD SPY',
    marker_color='lightblue',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal_spy,
    name='Signal SPY',
    marker_color='lightcoral',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram_spy,
    name='Histogram',
    marker_color=histogram_spy,
), row=3, col=1)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios BTC
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['BTC-USD'],
    line=dict(color='white'),
    name='Sharpe Ratio BTC',
), row=4, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['BTC-USD'],
    marker_color=sortino_ratio['BTC-USD'],
    name='Sortino Ratio BTC',
), row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top])*4,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino BTC > 3.5 & Sharpe > 2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom])*12,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < -2.0 & Shar < -2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_top],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe BTC > 2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['BTC-USD'][mask_btc_sortino_top]*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
                colorscale='PuRd',
                symbol='circle'),
    name='Sortino BTC > 3.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_bottom],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom])*3,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe BTC < -2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down])*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
                colorscale='YlOrRd',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_bottom])*3,
                color='green',
                symbol='circle'),
    name=f'Sortino BTC < -2.0'),
    row=4, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_up_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up_down])*5,
                color='lightblue',
                symbol='star'),
    name='Sort < -1.5 & Shar > 0'),
    row=4, col=1
)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios SPY
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['SPY'],
    line=dict(color='white'),
    name='Sharpe Ratio SPY',
), row=5, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['SPY'],
    marker_color=sortino_ratio['SPY'],
    name='Sortino Ratio SPY',
), row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_top])*3,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino SPY > 3.2 & Sharpe > 2.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom])*35,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < 0 & Shar < -0.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_top],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['SPY'][mask_spy_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe SPY > 2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['SPY'][mask_spy_sortino_top]*2,
                color=sortino_ratio['SPY'][mask_spy_sortino_top],
                colorscale='YlOrRd',
                symbol='circle'),
    name='Sortino SPY > 3.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_bottom],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['SPY'][mask_spy_sharpe_bottom])*8,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe SPY < -0.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_down],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_bottom])*12,
                color='green',
                symbol='circle'),
    name=f'Sortino SPY < 0'),
    row=5, col=1
)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B BTC-USD
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_b,
    name='BB %B BTC',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=6, col=1)

fig.update_layout(
    yaxis6=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B BTC"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=6, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=6, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=6, col=1)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B SPY
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_spy,
    name='BB %B SPY',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=7, col=1)

fig.update_layout(
    yaxis7=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B SPY"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=7, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=7, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=7, col=1)

# %% [markdown]
# ## Data Plotting RSI and Stochastic
# %%
fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi,
            name='RSI BTC',
            line=dict(color='white', width=1)
), row=8, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d,
            name='%D BTC',
            line=dict(color='red', width=1)
), row=8, col=1)

fig.add_hline(y=75, line_dash="dot", line_color="lightcoral", row=8, col=1)
fig.add_hline(y=50, line_dash="dot", line_color="gray", row=8, col=1)
fig.add_hline(y=25, line_dash="dot", line_color="lightgreen", row=8, col=1)

# %% [markdown]
# ## Data Plotting VIX
# %%

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=9, col=1)

# %% [markdown]
# ## Data Plotting End
# %%

fig.update_layout(
    title= f'Volatility: {trading_month_sortino} month / Sharpe Ratio Analysis: {trading_month_sharpe} months / Sortino Ratio Analysis: {trading_month_sortino} months',
    title_x=0.5,
    showlegend=False,
    height=1500,
    width=1150,
    yaxis_title='Price',
    yaxis_type='log',
    yaxis2_title='MACD BTC',
    yaxis3_title='MACD SPY',
    yaxis4_title='SHP&STO BTC',
    yaxis4_type='linear',
    yaxis5_title='SHP&STO SPY',
    yaxis5_type='linear',
    yaxis6_title='BB %B BTC',
    yaxis6_type='linear',
    yaxis7_title='BB %B SPY',
    yaxis7_type='linear',
    yaxis8_title='RSI & Stochastic',
    yaxis8_type='linear',
    yaxis9_title='VIX',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,


)

fig.show()